# Modul 6: Model Eksogen Cuaca (SARIMAX) & Multivariat VAR
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 6 (Minggu 6):** Mengintegrasikan variabel eksogen cuaca (temperatur & kelembaban) pada SARIMAX serta memodelkan interaksi multivariat (Tegangan, Arus, Beban) dengan VAR & *Impulse Response Function* (IRF).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

warnings.filterwarnings('ignore')
plt.rcParams['axes.grid'] = True

## 1. Simulasi Beban HVAC Gedung & Variabel Eksogen Cuaca

In [ ]:
rng = np.random.default_rng(66)
n = 24 * 14
idx = pd.date_range('2026-04-01', periods=n, freq='h')
hour = idx.hour

temp_c = 28 + 5.5 * np.sin(2 * np.pi * (hour - 8) / 24) + rng.normal(0, 1.0, n)
humidity = 75 - 12 * np.sin(2 * np.pi * (hour - 8) / 24) + rng.normal(0, 2.5, n)
# Elastisitas beban AC ~ 8.5 kW per °C
hvac_load = (
    120 + 8.5 * temp_c + 0.4 * humidity
    + 35 * np.sin(2 * np.pi * (hour - 7) / 24)
    + rng.normal(0, 4.5, n)
)
df = pd.DataFrame({'load': hvac_load, 'temp_c': temp_c, 'humidity': humidity}, index=idx)
train, test = df.iloc[:-24], df.iloc[-24:]

## 2. Komparasi SARIMA (Tanpa Cuaca) vs. SARIMAX (Dengan Eksogen Cuaca)

In [ ]:
m_base = SARIMAX(train['load'], order=(1, 0, 1), seasonal_order=(1, 1, 0, 24)).fit(disp=False)
m_exog = SARIMAX(
    train['load'], exog=train[['temp_c', 'humidity']],
    order=(1, 0, 1), seasonal_order=(1, 1, 0, 24)
).fit(disp=False)

fc_base = m_base.forecast(steps=24)
fc_exog = m_exog.forecast(steps=24, exog=test[['temp_c', 'humidity']])

rmse_base = np.sqrt(np.mean((test['load'] - fc_base) ** 2))
rmse_exog = np.sqrt(np.mean((test['load'] - fc_exog) ** 2))
print(f"Koefisien Eksogen SARIMAX -> temp_c: {m_exog.params['temp_c']:.3f} kW/°C | humidity: {m_exog.params['humidity']:.3f}")
print(f"RMSE SARIMA (tanpa cuaca) : {rmse_base:.2f} kW (AIC={m_base.aic:.1f})")
print(f"RMSE SARIMAX (dg cuaca)   : {rmse_exog:.2f} kW (AIC={m_exog.aic:.1f})")

## 3. Pemodelan Multivariat VAR & Impulse Response Function (IRF)

In [ ]:
# Simulasi sistem 2-kanal: Arus Beban (A) mempengaruhi Jatuh Tegangan Bus (V)
n_var = 250
arus = np.zeros(n_var)
tegangan = np.zeros(n_var)
for t in range(1, n_var):
    arus[t] = 0.65 * arus[t - 1] + rng.normal(0, 2.0)
    tegangan[t] = 0.50 * tegangan[t - 1] - 0.35 * arus[t - 1] + rng.normal(0, 0.5)

df_pq = pd.DataFrame({'arus': arus, 'tegangan': tegangan})
res_var = VAR(df_pq).fit(maxlags=8, ic='aic')
print(f"Ordo VAR terpilih (AIC): p = {res_var.k_ar}")

irf = res_var.irf(10)
irf.plot(impulse='arus', response='tegangan')
plt.suptitle('Impulse Response Function: Dampak Lonjakan Arus terhadap Tegangan Bus', y=1.02)
plt.show()

## 4. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Mengapa koefisien `temp_c` bertanda positif signifikan dan menurunkan RMSE/AIC pada SARIMAX?
2. **[C5]** Apa risiko praktis SARIMAX di lapangan jika ramalan temperatur BMKG 24 jam ke depan meleset $\pm 3^\circ\text{C}$? Uji dengan menambahkan derau pada `test['temp_c']`!